**1) Import Libraries**

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import scrapy
from typing import List, Dict, Any
from urllib.parse import urljoin
import re
import sqlite3
import json

**2) Getting Wikipedia Page**

In [2]:
# URL of the Wikipedia page
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; WikiScrapper/1.0; +https://example.com/bot)"
    )
}
def fetch_page(url: str) -> str:
    """Вернуть «сырой» HTML‑текст страницы."""
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text
try:
    soup = BeautifulSoup(fetch_page(url), 'html.parser')
    print("Successfully fetched Wikipedia page")
except requests.exceptions.RequestException as e:
    print(f"Error fetching page: {e}")

Successfully fetched Wikipedia page


**3) Finding table with films**

In [3]:
# Selector for the main table containing film data
table_selector = "#mw-content-text > div.mw-content-ltr.mw-parser-output > table"

try:
    film_table = soup.select_one(table_selector)
    if not film_table:
        raise ValueError("Table not found with the given selector")

    print("Found film table")
except Exception as e:
    print(f"Error finding table: {e}")


Found film table


**4) Getting data from table at corresponding references**

In [ ]:
def extract_film_data(table):
    film_data = [] #Array for films
    base_url = "https://en.wikipedia.org" #General URL of wiki

    for row in table.find_all('tr')[1:]: #Skip name row
        cols = row.find_all(['td', 'th'])
        try:
            title_tag = cols[2].find('a') #Column of title
            if not title_tag: #Just skip if empty
                continue

            title = title_tag.text.strip()
            film_url = base_url + title_tag['href']

            release_year_text = cols[4].text.strip()#Column of release year
            release_year = int(release_year_text) if release_year_text.isdigit() else None #Convert to int

            box_office = cols[3].text.strip()#Column of box office
            match = re.search(r'\$([\d,\.]+)', box_office)
    
            if match:
                box_office_value = match.group(1).replace(',', '')#Convert it to number

            director, country, time, language = extract_film_details(film_url)

            film_data.append({
                'title': title,
                'release_year': release_year,
                'director': director,
                'box_office': box_office_value,
                'country': country,
                'language': language,
                'running_time': time
            })
            print(f"Film '{title}' scrapped successfully")
            print(f"Parameters: {film_data[-1]}")
            time.sleep(1)

        except Exception as e:
            print(f"Scrapping of film '{title}' failed: {e}")
            continue

    return film_data

def extract_film_details(film_url):
    try:
        soup = BeautifulSoup(fetch_page(film_url), 'html.parser')

        infobox = soup.find('table')
        if not infobox:
            return "Unknown", "Unknown", "Unknown", "Unknown"

        director = "Unknown"
        countries = "Unknown"
        language = "Unknown"
        time = "Unknown"
        for row in infobox.find_all('tr'): #Go through all infobox
            header = row.find('th')
            if header:
                header_text = header.get_text(strip=True)
                
                if "Directed by" in header_text: #Get director of film
                    director_cell = row.find('td')
                    if director_cell:
                        director = director_cell.get_text(strip=True)
                        director = re.sub(r'\[\d+(?:-\d+)?\]', '', director)#delete references
                        list = re.findall(r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*', director)#convert it to array of directors
                        director = list[0]
                        for i in range(1, len(list)):
                            director += f", {list[i]}"#convert to comma separated string
                
                elif "Country" in header_text:#Case if 1 country
                    country_cell = row.find('td')
                    if country_cell:
                        countries = country_cell.get_text(strip=True)
                        countries = re.sub(r'\[\d+(?:-\d+)?\]', '', countries)#delete refernces
                        countries = re.findall(r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*', countries)[0]#get the only one country
                elif "Countries" in header_text:#Case for several countries
                    country_cell = row.find('td')
                    if country_cell:
                        countries = country_cell.get_text(strip=True)
                        countries = re.sub(r'\[\d+(?:-\d+)?\]', '', countries)
                        list = re.findall(r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*', countries)
                        
                        countries = list[0]
                        for i in range(1, len(list)):
                            countries += f", {list[i]}" #Convert it to comma separated string
                elif "Running time" in header_text:
                    time_cell = row.find('td')
                    if time_cell:
                        time = time_cell.get_text(strip=True)
                        time = re.sub(r'\[\d+(?:-\d+)?\]', '', time) #Delete references
                        time = time.strip().lower()
                        match = re.search(r'(\d+)\s*(?:min|minutes|minute)', time) #Delete minute word
                        if match:
                            time = int(match.group(1))#convert it to int
                elif "Language" in header_text:#Do same for lang like country
                    lang_cell = row.find('td')
                    if lang_cell:
                        language = lang_cell.get_text(strip=True)
                        language = re.sub(r'\[\d+(?:-\d+)?\]', '', language)
                        language = re.findall(r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*', language)[0]
        return director, countries, time, language #return all values

    except Exception as e:
        print(f"Error in film extracting: {e}")
        return "Unknown", "Unknown", "Unknown", "Unknown"

films = extract_film_data(film_table)
print(f"Extracted {len(films)} films")


Film 'Avatar' scrapped successfully
Parameters: {'title': 'Avatar', 'release_year': 2009, 'director': 'James Cameron', 'box_office': '2923710708', 'country': 'United States, United Kingdom', 'language': 'English', 'running_time': 162}
Scrapping of film 'Avatar' failed: 'int' object has no attribute 'sleep'
Film 'Avengers: Endgame' scrapped successfully
Parameters: {'title': 'Avengers: Endgame', 'release_year': 2019, 'director': 'Anthony Russo, Joe Russo', 'box_office': '2797501328', 'country': 'United States', 'language': 'English', 'running_time': 181}
Scrapping of film 'Avengers: Endgame' failed: 'int' object has no attribute 'sleep'
Film 'Avatar: The Way of Water' scrapped successfully
Parameters: {'title': 'Avatar: The Way of Water', 'release_year': 2022, 'director': 'James Cameron', 'box_office': '2334484620', 'country': 'United States', 'language': 'English', 'running_time': 192}
Scrapping of film 'Avatar: The Way of Water' failed: 'int' object has no attribute 'sleep'
Film 'Tita

**5) Converting to DataFrame**

In [ ]:
# Convert to DataFrame for easier manipulation
df = pd.DataFrame(films)
# Handle missing values
df = df.replace('', 'Unknown')

# Display all rows
print("\nSample of extracted data:")
df.head(50)


Sample of extracted data:


,title,release_year,director,box_office,country,language,running_time
0,Avatar,2009,James Cameron,2923710708,"United States, United Kingdom",English,162
1,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2797501328,United States,English,181
2,Avatar: The Way of Water,2022,James Cameron,2334484620,United States,English,192
3,Titanic,1997,James Cameron,2257906828,United States,English,195
4,Ne Zha 2,2025,Jiaozi,2215690000,China,Mandarin,144
5,Star Wars: The Force Awakens,2015,Abrams,2068223624,United States,English,138
6,Avengers: Infinity War,2018,"Anthony Russo, Joe Russo",2048359754,United States,English,149
7,Spider-Man: No Way Home,2021,Jon Watts,1922598800,United States,English,148
8,Zootopia 2,2025,"Jared Bush, Byron Howard",1866641191,United States,English,108
9,Inside Out 2,2024,Kelsey Mann,1698863816,United States,English,96


**6) Creating and Inserting Database**

In [40]:
def create_database(films):
    # Connect to SQLite database (creates if doesn't exist)
    conn = sqlite3.connect('films.db')
    cursor = conn.cursor()
    # Drop the table to avoid dublicates
    cursor.execute('''DROP TABLE IF EXISTS films''')
    # Create films table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS films (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        release_year INTEGER,
        director TEXT,
        box_office REAL,
        country TEXT,
        language TEXT,
        running_time INTEGER
    )
    ''')
    
    # Insert data
    for film in films:
        cursor.execute('''
        INSERT INTO films (title, release_year, director, box_office, country, language, running_time)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (film['title'], film['release_year'], film['director'], film['box_office'], film['country'], film['language'], film['running_time']))

    # Commit changes and close connection
    conn.commit()
    conn.close()
    print("Database created and data inserted successfully")

# Create database
create_database(films)


Database created and data inserted successfully


**7) Verifying the database**

In [51]:
def verify_database():
    conn = sqlite3.connect('films.db') #open database
    cursor = conn.cursor()

    # Get count of records
    cursor.execute('SELECT COUNT(*) FROM films')
    count = cursor.fetchone()[0]
    print(f"\nDatabase contains {count} films")

    # Show first 5 records
    cursor.execute('SELECT * FROM films')
    print("\nRecords in database:")
    for row in cursor.fetchall():
        print(row)

    conn.close()

# Verify data
verify_database()



Database contains 50 films

Records in database:
(1, 'Avatar', 2009, 'James Cameron', 2923710708.0, 'United States, United Kingdom', 'English', 162)
(2, 'Avengers: Endgame', 2019, 'Anthony Russo, Joe Russo', 2797501328.0, 'United States', 'English', 181)
(3, 'Avatar: The Way of Water', 2022, 'James Cameron', 2334484620.0, 'United States', 'English', 192)
(4, 'Titanic', 1997, 'James Cameron', 2257906828.0, 'United States', 'English', 195)
(5, 'Ne Zha 2', 2025, 'Jiaozi', 2215690000.0, 'China', 'Mandarin', 144)
(6, 'Star Wars: The Force Awakens', 2015, 'Abrams', 2068223624.0, 'United States', 'English', 138)
(7, 'Avengers: Infinity War', 2018, 'Anthony Russo, Joe Russo', 2048359754.0, 'United States', 'English', 149)
(8, 'Spider-Man: No Way Home', 2021, 'Jon Watts', 1922598800.0, 'United States', 'English', 148)
(9, 'Zootopia 2', 2025, 'Jared Bush, Byron Howard', 1866641191.0, 'United States', 'English', 108)
(10, 'Inside Out 2', 2024, 'Kelsey Mann', 1698863816.0, 'United States', 'Engli

**8) Converting data into json format for webpage**

In [ ]:
def export_database_to_json(db_path='films.db', json_path='films_data.json'):
    # Export the films database to a JSON file for use with GitHub Pages
    # Connect to the database
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row  # This allows us to access columns by name
    
    # Query all films
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM films ORDER BY box_office DESC')
    
    # Fetch all rows and convert to list of dictionaries
    films = []
    for row in cursor.fetchall():
        films.append({
            'id': row['id'],
            'title': row['title'],
            'release_year': row['release_year'],
            'director': row['director'],
            'box_office': row['box_office'],
            'country': row['country'],
            'language': row['language'],
            'running_time': row['running_time']
        })
    
    # Write to JSON file
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(films, f, indent=2, ensure_ascii=False)
    
    print(f"Exported {len(films)} films to {json_path}")
    print(films)
    # Print sample data
    if films:
        print("\nSample data:")
        print(f"  Title: {films[0]['title']}")
        print(f"  Year: {films[0]['release_year']}")
        print(f"  Director: {films[0]['director']}")
        print(f"  Box Office: ${films[0]['box_office']:,}")
        print(f"  Country: {films[0]['country']}")
        print(f"  Language: {films[0]['language']}")
        print(f"  Running time: {films[0]['running_time']} minutes")
    
    conn.close()
    return films

# Run the export
if __name__ == "__main__":
    export_database_to_json()

Exported 50 films to films_data.json
[{'id': 1, 'title': 'Avatar', 'release_year': 2009, 'director': 'James Cameron', 'box_office': 2923710708.0, 'country': 'United States, United Kingdom', 'language': 'English', 'running_time': 162}, {'id': 2, 'title': 'Avengers: Endgame', 'release_year': 2019, 'director': 'Anthony Russo, Joe Russo', 'box_office': 2797501328.0, 'country': 'United States', 'language': 'English', 'running_time': 181}, {'id': 3, 'title': 'Avatar: The Way of Water', 'release_year': 2022, 'director': 'James Cameron', 'box_office': 2334484620.0, 'country': 'United States', 'language': 'English', 'running_time': 192}, {'id': 4, 'title': 'Titanic', 'release_year': 1997, 'director': 'James Cameron', 'box_office': 2257906828.0, 'country': 'United States', 'language': 'English', 'running_time': 195}, {'id': 5, 'title': 'Ne Zha 2', 'release_year': 2025, 'director': 'Jiaozi', 'box_office': 2215690000.0, 'country': 'China', 'language': 'Mandarin', 'running_time': 144}, {'id': 6, 't